## Project 2     

# Budget-Constrained Stock Selection with 0/1 Knapsack on S&P 500 Data

### MSML606
### Team Composition: Individual Project
### Author: Chenhongshu Yu
### UID: 116610971

### Statement 
- The use of external resources is clearly cited. Please refer to the bold words in my submission. 
- If no citation is found for any specific part of my submission, it means "No external sources were used; all ideas are my own or from course lecture slides".
- The core logic and initial approach are manually authored and typed by the author Chenhongshu Yu.

## Problem
US stock market is one of the most vabriant market in the world. It is full with opportunities and also the risk. In real investing, even though many stocks look attractive, it is usually not reasonable to buy all of them, because doing so may create a portfolio that is too risky.       

This project studies how to choose the best selection (basket) of stocks when an investor cannot take unlimited risk. I model this as a budget-constrained stock selection problem: each stock has a potential benefit and a risk cost, and the goal is to select the combination of stocks that gives the highest total benefit while staying under a fixed risk budget.

To be more specific, each stock will be treated as one item in a **0/1 knapsack problem**. For the simple measurement of the risk, the key points are set as:   

- each stock = one item
- value = a momentum-based score derived from recent price performance
- weight = the stock’s 20-day volatility
- capacity = fixed total risk budget

**The Question:**       
Which subset of stocks should be selected to maximize total attractiveness while keeping total risk under control?

## Dataset

Kaggle S&P 500 dataset: https://www.kaggle.com/datasets/jacksaleeby/s-and-p500-historical-data

I choose a public Kaggle dataset containing daily OHLCV stock data for current S&P 500 companies. The dataset includes historical daily prices and trading volume for 472 current S&P 500 companies, with more than 2.7 million rows of data spanning January 2000 to February 2026. This dataset is suitable because it provides the price history needed to compute momentum-based scores and rolling volatility measures for each stock.             

From the dataset, I will use daily closing prices and possibly trading volume to compute features such as recent returns and 20-day volatility. These features will then be converted into the value and weight inputs required by the optimization algorithm.

## Algorithm and Usage

The main algorithm will be **0/1 knapsack** solved with **dynamic programming (DP)**. For each decision date, every stock is represented by:        

- value = a momentum-based score derived from recent price performance      
- weight = the stock’s 20-day volatility        
- capacity = the fixed maximum total risk budget        

The dynamic programming algorithm will search for the subset of stocks that maximizes the total value while ensuring that the sum of their risk weights stays below a fixed risk budget. This gives an exact solution under the project’s simplified portfolio model.

As a comparison baseline, I will also implement a **greedy method** that ranks stocks by their value-to-risk ratio and selects them in that order until the risk budget is reached. This baseline is faster and easier to explain, but it may not always find the optimal portfolio. Comparing the greedy approach with dynamic programming will show the benefit of using a more principled algorithms method.

## Necessary Functions
### 1. Momentum-Based Value Function

For stock \(i\) on day \(t\), let \(P_i(t)\) be its closing price.

The 5-day return is:

$$
r_{5,i}(t)=\frac{P_i(t)-P_i(t-5)}{P_i(t-5)}
$$

The 20-day return is:

$$
r_{20,i}(t)=\frac{P_i(t)-P_i(t-20)}{P_i(t-20)}
$$

Then the stock's value is defined as a weighted momentum score:

$$
v_i=\max\left(0,\;0.6\,r_{5,i}(t)+0.4\,r_{20,i}(t)\right)
$$

This means stocks with stronger recent momentum receive higher value, while stocks with negative momentum are assigned value 0.

### 2. Risk Weight Function

First, define the daily return of stock \(i\) on day \(\tau\) as:

$$
d_{i,\tau}=\frac{P_i(\tau)-P_i(\tau-1)}{P_i(\tau-1)}
$$

Then compute the average daily return over the past 20 days:

$$
\bar d_i=\frac{1}{20}\sum_{k=0}^{19} d_{i,t-k}
$$

The 20-day volatility is used as the stock's weight:

$$
w_i=\sigma_i=\sqrt{\frac{1}{20}\sum_{k=0}^{19}\left(d_{i,t-k}-\bar d_i\right)^2}
$$

A higher \(w_i\) means the stock is more volatile and therefore costs more of the portfolio's risk budget.

### 3. Decision Variable

For each stock \(i\), define a binary selection variable:

$$
x_i \in \{0,1\}
$$

where:

- \(x_i=1\): select stock \(i\)
- \(x_i=0\): do not select stock \(i\)

### 4. 0/1 Knapsack Optimization Problem

The portfolio selection problem is:

$$
\max \sum_{i=1}^{n} v_i x_i
$$

subject to:

$$
\sum_{i=1}^{n} w_i x_i \leq W
$$

$$
x_i \in \{0,1\}, \quad i=1,\dots,n
$$

where:

- \(n\) = number of candidate stocks
- \(v_i\) = momentum-based value of stock \(i\)
- \(w_i\) = 20-day volatility of stock \(i\)
- \(W\) = total risk budget

This finds the best basket of stocks under a risk limit.

### 5. Dynamic Programming Solution

Let \(DP[i][c]\) represent the maximum total value achievable using the first \(i\) stocks with remaining capacity \(c\).

The recurrence is:

$$
DP[i][c]=
\begin{cases}
DP[i-1][c], & \text{if } w_i > c \\\\
\max\left(DP[i-1][c],\;DP[i-1][c-w_i]+v_i\right), & \text{if } w_i \leq c
\end{cases}
$$

with base case:

$$
DP[0][c]=0
$$

This dynamic programming method gives the **exact optimal solution** for the knapsack formulation.

### 6. Greedy Baseline

As a baseline, stocks can also be ranked by their value-to-risk ratio:

$$
\rho_i=\frac{v_i}{w_i}
$$

The greedy method sorts all stocks by \(\rho_i\) from highest to lowest and keeps selecting them until the total risk budget is reached.

This baseline is simple and fast, but it does **not always produce the optimal solution**, which is why comparing it with dynamic programming is meaningful.

## FallBack Plan
If the stock-selection project becomes too difficult to implement within the project timeline, the backup project will be **flight route planning using graph algorithms**. In this version, airports are modeled as nodes and direct flights as edges in a graph, and algorithms such as **BFS** and **Dijkstra’s algorithm** can be used to find routes with the fewest stops or the shortest total distance. This fallback still uses a real-world dataset from kaggle: https://www.kaggle.com/datasets/elmoallistair/airlines-airport-and-routes
